In [ ]:
import requests
import pandas as pd
import time

# Konfigurasi API dan Stasiun
API_TOKEN = "apify_api_3zLbt6cWLZstk518kjnEAOyLGYtSye3vKUV8"

# Endpoint resmi Apify
API_URL = f"https://api.apify.com/v2/acts/compass~crawler-google-places/runs?token={API_TOKEN}"

print("Menyiapkan payload data 4 stasiun...")
payload = {
    "startUrls": [
        {"url": "https://www.google.com/maps/place/Stasiun+Lama+Gubeng/@-7.2653362,112.7496453,17z/data=!4m12!1m2!2m1!1sStasiun+Gubeng!3m8!1s0x2dd7f9679ca28edb:0x2ab7b703ea65869f!8m2!3d-7.2653365!4d112.7519634!9m1!1b1!15sCg5TdGFzaXVuIEd1YmVuZ1oQIg5zdGFzaXVuIGd1YmVuZ5IBHXRyYW5zcG9ydGF0aW9uX2VzY29ydF9zZXJ2aWNlmgEkQ2hkRFNVaE5NRzluUzBWSlEwRm5TVU5ZZHkxbGIzcG5SUkFC4AEA-gEECAAQMQ!16s%2Fg%2F11h4c8m7r9?entry=ttu&g_ep=EgoyMDI2MDQyOS4wIKXMDSoASAFQAw%3D%3D"},
        {"url": "https://www.google.com/maps/place/Surabaya+Pasar+Turi+Train+Station/@-7.248272,112.7297697,18z/data=!4m8!3m7!1s0x2dd7f9476e657037:0x520b4ad5fa4c056d!8m2!3d-7.248256!4d112.7310733!9m1!1b1!16s%2Fm%2F05zw_gj?entry=ttu&g_ep=EgoyMDI2MDQyOS4wIKXMDSoASAFQAw%3D%3D"},
        {"url": "https://www.google.com/maps/place/Surabaya+Kota/@-7.24299,112.7389751,17z/data=!4m8!3m7!1s0x2dd7fcb3c1d50a4b:0xa8a6ab495b6fe689!8m2!3d-7.24299!4d112.74155!9m1!1b1!16s%2Fm%2F0y6dl4w?entry=ttu&g_ep=EgoyMDI2MDQyOS4wIKXMDSoASAFQAw%3D%3D"},
        {"url": "https://www.google.com/maps/place/Stasiun+Wonokromo/@-7.3023043,112.735872,17z/data=!4m12!1m2!2m1!1sstasiun+wonokromo!3m8!1s0x2dd7fb9f49626167:0xa88575f9052f43da!8m2!3d-7.3027335!4d112.7378038!9m1!1b1!15sChFzdGFzaXVuIHdvbm9rcm9tb5IBF2xvZ2ljYWxfdHJhbnNpdF9zdGF0aW9u4AEA!16s%2Fm%2F080ljkg?entry=ttu&g_ep=EgoyMDI2MDQyOS4wIKXMDSoASAFQAw%3D%3D"}
    ],
    "maxReviews": 15000,
    "language": "id",
    "reviewsSort": "newest",
    "scrapeReviewsPersonalData": True
}

# Pemanggilaan API Apify
response = requests.post(API_URL, json=payload)

if response.status_code == 201:
    run_data = response.json()
    run_id = run_data['data']['id']
    default_dataset_id = run_data['data']['defaultDatasetId']
    print(f"API berhasil dipanggil! Tugas bot sedang berjalan dengan ID: {run_id}")

    status = "RUNNING"

    while status not in ["SUCCEEDED", "FAILED", "ABORTED"]:
        time.sleep(15) 
        cek_status_url = f"https://api.apify.com/v2/actor-runs/{run_id}?token={API_TOKEN}"
        status_response = requests.get(cek_status_url).json()
        status = status_response['data']['status']
        print(f"Status saat ini: {status}")

# Hasil JSONnya di convert ke pandas
    if status == "SUCCEEDED":
        print("\nProses scraping berhasil! Mengunduh dataset mentah (Raw Data) ke MacBook...")
        dataset_url = f"https://api.apify.com/v2/datasets/{default_dataset_id}/items?token={API_TOKEN}"
        hasil_data = requests.get(dataset_url).json()

        semua_ulasan = []

        for tempat in hasil_data:
            nama_stasiun = tempat.get('title', 'Stasiun Tidak Diketahui')

            if 'reviews' in tempat and tempat['reviews'] is not None:
                for ulasan in tempat['reviews']:
                    semua_ulasan.append({
                        'Station Name': nama_stasiun,
                        'Reviewer Name': ulasan.get('name', ''),
                        'Rating Star': ulasan.get('stars', ''),
                        'Review Text': ulasan.get('text', ''),
                        'Review Date': ulasan.get('publishedAtDate', ''),
                        'User Contribution Level': ulasan.get('reviewerNumberOfReviews', '')
                    })

        # Konversi array JSON ke DataFrame Pandas
        df = pd.DataFrame(semua_ulasan)

        # Simpan ke CSV
        nama_file_csv = "Dataset_Review_Stasiun_Surabaya_RAW.csv"
        df.to_csv(nama_file_csv, index=False)
        print(f"\nMenyimpan {len(df)} baris ulasan mentah ke dalam file {nama_file_csv}.")
        df.head()

    else:
        print(f"Scraping berhenti dengan status: {status}")

else:
    print(f"Gagal memanggil API. Kode Error: {response.status_code}")
    print(f"Pesan: {response.text}")
    print("Pastikan API Token Anda sudah benar!")